<a href="https://colab.research.google.com/github/averkina/test1/blob/main/Test1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -O shape_predictor_68_face_landmarks.dat.bz2 http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 && bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

--2026-05-17 08:47:42--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-05-17 08:47:43--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  39.5MB/s    in 1.5s    

2026-05-17 08:47:44 (39.5 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



In [ ]:
#В наборе shape_predictor_68_face_landmark самыми верхними точками являются юрови, то есть точек лба нету.
#Поэтому в коде я достраиваю лоб, рассчитав координаты его точек
#Беру расстояние от переносицы до подбородка и отмеряю 60% от этого расстояния вверх до бровей

In [8]:
import os
import cv2
import dlib
import numpy as np
import glob

files = glob.glob("*.png") + glob.glob("*.jpg") + glob.glob("*.jpeg")
INPUT_IM = files[0]
OUTPUT_IM = "output.png"
PREDICTOR = "shape_predictor_68_face_landmarks.dat"

predictor = dlib.shape_predictor(PREDICTOR)
detector = dlib.get_frontal_face_detector()

image = cv2.imread(INPUT_IM)
if image is None:
    raise FileNotFoundError(f"Не удалось открыть: {INPUT_IM}")

height, width = image.shape[:2]
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

faces = detector(gray)
print("Найдено лиц:", len(faces))
if len(faces) == 0:
    raise ValueError("Лицо на фото не найдено")

final_mask = np.zeros((height, width), dtype=np.uint8)

for face in faces:
    landmarks = predictor(gray, face)
    points = np.array([(landmarks.part(i).x, landmarks.part(i).y) for i in range(68)], dtype=np.int32)

    # --- точки лба ---
    chin_y = points[8][1]
    nose_y = points[27][1]
    forehead_height = int((chin_y - nose_y) * 0.55) #взяла 55 процентов

    # Строю 4 искусственные точки лба
    # Левый висок, левый центр лба, правый центр лба, правый висок
    forehead_p1 = [points[17][0], max(0, points[17][1] - forehead_height)]
    forehead_p2 = [points[20][0], max(0, points[20][1] - forehead_height)]
    forehead_p3 = [points[23][0], max(0, points[23][1] - forehead_height)]
    forehead_p4 = [points[26][0], max(0, points[26][1] - forehead_height)]

    forehead_points = np.array([forehead_p1, forehead_p2, forehead_p3, forehead_p4], dtype=np.int32)

    # Соединяю контур челюсти (0-16) и искусственный лоб
    jaw_contour = points[0:17]
    face_contour = np.vstack([jaw_contour, forehead_points[::-1]])

    face_mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(face_mask, [face_contour], 255)

    # Вырезаю глаза и рот, так как спрашивается в задании кожный покров
    for region in (points[36:42], points[42:48], points[48:60]):
        cv2.fillPoly(face_mask, [cv2.convexHull(region)], 0)

    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    lower = np.array([0, 133, 77], dtype=np.uint8)
    upper = np.array([255, 173, 127], dtype=np.uint8)
    skin_color_mask = cv2.inRange(ycrcb, lower, upper)

    skin_mask = cv2.bitwise_and(face_mask, skin_color_mask)

    if cv2.countNonZero(skin_mask) < 0.15 * cv2.countNonZero(face_mask):
        skin_mask = face_mask

    kernel = np.ones((5, 5), np.uint8)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel)
    skin_mask = cv2.GaussianBlur(skin_mask, (3, 3), 0)

    final_mask = cv2.bitwise_or(final_mask, skin_mask)

result = cv2.bitwise_and(image, image, mask=final_mask)

ext = os.path.splitext(INPUT_IM)[1].lower()
if ext in (".jpg", ".jpeg"):
    out_path = os.path.splitext(OUTPUT_IM)[0] + ext
    cv2.imwrite(out_path, result, [cv2.IMWRITE_JPEG_QUALITY, 95])
else:
    out_path = os.path.splitext(OUTPUT_IM)[0] + ".png"
    cv2.imwrite(out_path, result)

print(f"Сохранено: {out_path}")

Найдено лиц: 2
Сохранено: output.jpg, размер 862x528
